# Hybrid RAG — Full Demonstration

Adaptive vector + graph + vectorless RAG with two **LangGraph** agents, run end-to-end.

**What this notebook demonstrates**
1. Load real documents from the `docs/` folder (§3)
2. **Agent 1** — full-document **entity gate** + **router** (entity-dense → graph, else → vector), **batch** ingest (§4)
3. The **routing catalog** that records where each doc went (§5)
4. **Knowledge-graph** + **PageIndex-tree** visualizations (§6–7)
5. **Agent 2** — the **CRAG** loop: evaluate → correct / ambiguous / incorrect → generate, with LLM-call counts (§8)
6. **Vectorless** correction + **multi-document scoping** (§9)
7. The **dual-store** toggle (§10)
8. The same pipeline against your **real Neptune** cluster (§11–13)

Sections 0–10 run **fully offline with fakes** (no credentials). 11–13 use your real Neptune endpoint.

> The agents are dependency-injected: the exact same code runs with fake clients or the real
> LMaaS/DIS/Neptune clients — only the objects passed in change.

## 0. Install dependencies (run once per kernel)

Both agents are LangGraph graphs, so **`langgraph` must be installed in this kernel.** If a later
cell raises `ModuleNotFoundError: No module named 'langgraph'`, run this, then **restart the kernel
(Kernel → Restart)** and re-run — a package pip-installed into a running kernel isn't always visible
to the already-started interpreter.

In [ ]:
%pip install -q -r ../requirements.txt
import importlib.util
print("langgraph present:", importlib.util.find_spec("langgraph") is not None)
print("(if False, or a later import fails, restart the kernel and re-run)")

## 1. Setup

In [ ]:
import os, sys, tempfile
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))  # make hybridrag importable

from IPython.display import HTML
from hybridrag import visualize
from hybridrag.loader import load_documents
from hybridrag.fakes import FakeLMaaS, FakeDIS, FakeGraph
from hybridrag.catalog import RoutingCatalog
from hybridrag.ingest import IngestAgent
from hybridrag.crag import CRAGQueryAgent
print("imports OK — offline mode (fakes)")

## 2. Build the agents (offline fakes **or** real LMaaS)

Same constructor whether the LLM is fake or real — the injection point that makes the whole
system testable without credentials. The cell below auto-selects **real LMaaS** when
`IDAM_APP_CLIENT_ID` / `IDAM_APP_CLIENT_SECRET` are set in the kernel environment, otherwise it
falls back to the offline fake. DIS + Graph stay fake here. `min_graph_entities=5` is the graph gate.

In [ ]:
# --- LLM backend: real LMaaS when creds are present, offline fake otherwise ---
USE_REAL_LMAAS = bool(os.environ.get("IDAM_APP_CLIENT_ID") and os.environ.get("IDAM_APP_CLIENT_SECRET"))

if USE_REAL_LMAAS:
    from hybridrag import config
    from hybridrag.clients import LMaaSClient
    config.LMAAS_ENDPOINT          = "https://lmaas-integ-int.ailab.gehealthcare.net"
    config.LMAAS_AUDIENCE          = "0_b2dJB20TBhxzLIHCMzSG4RiQYa"
    config.LMAAS_API_VERSION       = "2025-04-01-preview"
    config.LMAAS_DEPLOYMENT_STRONG = "integ-gpt-5.2-2025-12-11"   # final generation
    config.LMAAS_DEPLOYMENT_CHEAP  = "integ-gpt-5.2-2025-12-11"   # evaluator/nav/refine (use a mini deployment if you have one)
    config.IDAM_TOKEN_ENDPOINT     = "https://idam.gehealthcloud.io/oauth2/token"
    config.IDAM_APP_CLIENT_ID      = os.environ["IDAM_APP_CLIENT_ID"]
    config.IDAM_APP_CLIENT_SECRET  = os.environ["IDAM_APP_CLIENT_SECRET"]
    # config.LMAAS_VERIFY_SSL = False   # uncomment if you hit SSLCertVerificationError on the GE network
    lmaas = LMaaSClient()
    print("LLM backend: REAL LMaaS ->", config.LMAAS_ENDPOINT)
else:
    lmaas = FakeLMaaS()
    print("LLM backend: offline FAKE (set IDAM_APP_CLIENT_ID + IDAM_APP_CLIENT_SECRET env vars for real LMaaS)")

dis, graph = FakeDIS(), FakeGraph()

cat_path = os.path.join(tempfile.gettempdir(), "hybridrag_demo_catalog.json")
if os.path.exists(cat_path):
    os.remove(cat_path)
catalog = RoutingCatalog(path=cat_path)

ingest = IngestAgent(lmaas, dis, graph, catalog=catalog, collection="demo", min_graph_entities=5)
query  = CRAGQueryAgent(lmaas, dis, graph, catalog=catalog, collection="demo")
print("Agent 1 (ingest) and Agent 2 (CRAG) ready")

## 3. Load documents from `docs/`

Drop your own `.md` / `.txt` / `.pdf` files into `docs/` to use them here. Two samples ship:
an **entity-dense** service manual and a **structured** field handbook (good for vectorless).

In [ ]:
docs = load_documents(os.path.join("..", "docs"))
for d in docs:
    print(f"  {d['doc_id']:26s} {len(d['text'].split()):4d} words   {d['title']}")

## 4. Agent 1 — entity gate + router + batch ingest

`ingest_documents` runs the LangGraph router over each doc: it scans the **whole document** for
distinct entities and routes **entity-dense docs to the graph**, everything else to the vector
store. Watch the `entities` count drive the `graph`/`vector` decision.

In [ ]:
entries = ingest.ingest_documents(docs)
print(f"{'doc_id':26s} {'graph':6s} {'vector':7s} {'entities':9s} density  domain")
for e in entries:
    print(f"{e.doc_id:26s} {str(e.in_graph):6s} {str(e.in_vector):7s} "
          f"{e.meta['entity_count']:<9d} {e.meta['density']:<7} {e.domain}")

## 5. The routing catalog

The linchpin: it records which store each document went to, so Agent 2 routes queries instead of
guessing. Agent 2 reads this at query time.

In [ ]:
for e in catalog.all():
    print(f"{e.doc_id:26s} in_graph={e.in_graph} in_vector={e.in_vector}")
    print(f"    entities: {e.entities[:6]}")

## 6. Visualize the knowledge graph

`Document → Section → Chunk`, as Agent 1 wrote it. (Change `DOC` to any ingested doc_id.)

In [ ]:
DOC = "imaging_service_manual"
tree = graph.get_tree([DOC])
HTML(visualize.iframe_srcdoc(visualize.graph_html(tree, f"Knowledge Graph — {DOC}"), 560))

## 7. Visualize the PageIndex tree

The hierarchical table-of-contents that **vectorless** retrieval reasons over. Sub-sections nest
by their dotted heading numbers.

In [ ]:
HTML(visualize.iframe_srcdoc(visualize.pageindex_html(tree, f"PageIndex Tree — {DOC}"), 560))

## 8. Agent 2 — the CRAG query loop

For each query: **route → retrieve → evaluate → (correct | ambiguous | incorrect) → generate**.
Watch `verdict`, which `sources` contributed, and `LLM calls` (only the final answer uses the
strong model; evaluator/nav/refine use the cheap tier).

In [ ]:
def ask(q):
    r = query.run(q)
    print(f"Q: {q}")
    print(f"   verdict : {r.verdict.label} (score={r.verdict.score:.2f})")
    print(f"   sources : {r.sources}")
    print(f"   LLM calls: {r.llm_calls}")
    for step in r.trace:
        print(f"     - {step}")
    print(f"   answer  : {r.answer[:170]}\n")

ask("Error Code E-204 Collimator Alignment Fault")   # graph body hit -> CORRECT
ask("Escalation and On-Call Rotation")               # body misses, heading matches -> INCORRECT -> vectorless
ask("Detector Calibration escalation")               # partial body + other-doc heading -> AMBIGUOUS -> fuse

## 9. Vectorless correction + multi-document scoping

When primary retrieval is weak, Agent 2 corrects with **vectorless** navigation over the section
tree — scoped to just the documents the catalog says are relevant, so it reads a focused
table-of-contents even across a large corpus.

In [ ]:
print("query -> scoped candidate documents (fed to vectorless):")
for q in ["escalation on-call rotation", "collimator alignment error code",
          "smart reading protocol layout"]:
    print(f"  {q!r:42s} -> {catalog.candidate_doc_ids(q)}")

## 10. Storage policy toggle

By default an entity-dense doc is **graph-only** (not duplicated as vectors). Flip
`STORE_GRAPH_DOCS_IN_VECTOR` to dual-store it — useful if you also want semantic vector recall on
graph docs.

In [ ]:
from hybridrag import config

config.STORE_GRAPH_DOCS_IN_VECTOR = True          # dual-store
g2, d2 = FakeGraph(), FakeDIS()
c2 = RoutingCatalog(path=os.path.join(tempfile.gettempdir(), "hybridrag_dual.json"))
if os.path.exists(c2.path): os.remove(c2.path); c2 = RoutingCatalog(path=c2.path)
ia = IngestAgent(FakeLMaaS(), d2, g2, catalog=c2, collection="dual", min_graph_entities=5)
e = ia.ingest_documents(docs[:1])[0]
print(f"dual-store ON  -> {e.doc_id}: graph={e.in_graph} vector={e.in_vector}")

config.STORE_GRAPH_DOCS_IN_VECTOR = False         # reset default for the rest of the notebook
print("reset to graph-only default")

## 11. Switch to the real Neptune cluster

Everything above ran on fakes. The rest talks to your **real Neptune** via `GraphClient`
(SigV4-signed Gremlin), keeping LMaaS/DIS faked — so you only need the **Neptune endpoint**.

Run on a host with network access to the cluster (this SageMaker notebook in the Neptune VPC) and
an IAM role permitted to reach Neptune.

In [ ]:
from hybridrag import config
config.NEPTUNE_ENDPOINT = "REPLACE_ME"     # <-- your cluster (writer) endpoint, no https://, no :8182
config.NEPTUNE_PORT = 8182
config.NEPTUNE_USE_IAM = True
config.AWS_REGION = "us-east-1"            # <-- your region

from hybridrag.clients import GraphClient
neptune = GraphClient()
assert config.NEPTUNE_ENDPOINT != "REPLACE_ME", "Set config.NEPTUNE_ENDPOINT above first!"
print("connectivity:", neptune.run_gremlin("g.V().limit(1).count()"))

## 12. Agent 1 → real Neptune (the graph build is real)

Re-ingests the **same `docs/`** — but now the graph is written to your real cluster. Structuring
and the Neptune writes are 100% real; the entity-gate uses the heuristic stand-in for LMaaS (no
creds). Swap `FakeLMaaS()` → `LMaaSClient()` to make it real LLM-driven.

In [ ]:
from hybridrag.ingest import IngestAgent
from hybridrag.fakes import FakeLMaaS, FakeDIS
from hybridrag.catalog import RoutingCatalog

ncat = RoutingCatalog(path=os.path.join(tempfile.gettempdir(), "hybridrag_neptune.json"))
if os.path.exists(ncat.path): os.remove(ncat.path); ncat = RoutingCatalog(path=ncat.path)
ingest_real = IngestAgent(FakeLMaaS(), FakeDIS(), neptune,
                          catalog=ncat, collection="hybrid-docs", min_graph_entities=5)
for e in ingest_real.ingest_documents(docs):
    print(f"  {e.doc_id:26s} graph={e.in_graph} entities={e.meta['entity_count']}")

### 12.1 Visualize a document from LIVE Neptune

In [ ]:
DOC = "imaging_service_manual"
live_tree = neptune.get_tree([DOC])
print(DOC, "sections:", [s["heading"] for s in live_tree[0]["sections"]])
HTML(visualize.iframe_srcdoc(visualize.graph_html(live_tree, f"LIVE Neptune — {DOC}"), 560))

In [ ]:
HTML(visualize.iframe_srcdoc(visualize.pageindex_html(live_tree, f"LIVE PageIndex — {DOC}"), 560))

## 13. Vectorless retrieval over real Neptune

Reads section headings from **live Neptune**, navigates to the relevant section, and fetches its
text from the cluster — the exact corrective path Agent 2 uses, against your real graph.

In [ ]:
from hybridrag.retrieval import vectorless_retrieve
from hybridrag.fakes import FakeLMaaS

for q in ["Error Code E-204 Collimator Alignment Fault", "Escalation and On-Call Rotation"]:
    res = vectorless_retrieve(FakeLMaaS(), neptune, q, doc_ids=ncat.candidate_doc_ids(q))
    print(f"Q: {q}")
    print(f"   sections: {[c['heading'] for c in res.chunks]}")
    print(f"   text: {res.text[:220]}\n")

### 13.1 (optional) Clean up what this wrote to Neptune

In [ ]:
# for did in ("imaging_service_manual", "field_service_handbook"):
#     neptune.run_gremlin(f"g.V().has('docId','doc_{did}').drop()")   # sections + chunks
#     neptune.run_gremlin(f"g.V('doc_{did}').drop()")                 # the Document node
# print("cleaned up")

# Full Agent 2 (CRAG) over real Neptune:
#   qagent = CRAGQueryAgent(FakeLMaaS(), FakeDIS(), neptune, catalog=ncat, collection="hybrid-docs")
#   qagent.run("...")   # note: graph keyword-search needs Neptune full-text search enabled